# Supervisor notes: sorting, the split, and a check that the sorting works
The README asks for three things: sort the notes into a small set of reasons for the extra hours, split the overtime into hours the client asked for versus operational failures (and say where that is concentrated), and check whether the sorting is any good with no answer sheet.

**Approach in one paragraph.** The notes are not free text in the usual sense: about 2,100 notes collapse to roughly 60 repeated sentences once typos, surnames and times are ignored. So I (1) collapse the notes to templates, (2) sort them with typo-tolerant keyword rules (Method A) and, independently, with unsupervised clusters whose clusters I label once by reading (Method B), and (3) check both against a **blind hand-labelled sample** that I wrote before building either method (`evaluation/labels/notes_blind_sample.csv`). I did not use a language model: there are few templates, a model adds cost and an error rate I cannot explain, and training a classifier would need labels that only my own rules could supply, which is circular.

**What differs from a notes-only analysis.** `shift_notes.csv` has a `shift_id`, so every note joins to `shifts.csv` and so has a site, a date, an employee and a number of hours. That lets the split be weighted by hours, not just counted, and lets concentration be checked by site as well as by supervisor.

**Labels supplied with the task.** `data/shift_notes_labelled.csv` has a category for every note but was produced by someone else's rules or model, so it is not an independent answer key. It is only used at the end as a third opinion (agreement is not accuracy).

In [1]:
import sys, re
sys.path.insert(0, "..") if False else None
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from rapidfuzz import fuzz, process
from scipy import stats

from evaluation.notes_check import (normalise, surname_set, mask, template_clusters, draw_blind_samples, accuracy_table)
from src.loader import load_shifts, load_holidays
from src.hours import parse_shifts, attribute_hours

pd.set_option("display.width", 220); pd.set_option("display.max_columns", 30); pd.set_option("display.max_colwidth", 90)
notes = pd.read_csv("data/shift_notes.csv", keep_default_na=False)
employees = pd.read_csv("data/employees.csv")
supplied = pd.read_csv("data/shift_notes_labelled.csv", keep_default_na=False)
print(notes.shape, "| supervisors:", notes.logged_by.nunique(), "| notes per supervisor:", notes.logged_by.value_counts().min(), "to", notes.logged_by.value_counts().max())

(2117, 3) | supervisors: 24 | notes per supervisor: 65 to 100


## 1. Look at the data first

In [2]:
shifts = parse_shifts(load_shifts("data/shifts.csv"))
holidays = load_holidays("data/public_holidays.csv")
segments = attribute_hours(shifts, holidays)

print("notes whose shift_id is in shifts.csv:", notes.shift_id.isin(shifts.shift_id).sum(), "of", len(notes))
print("share of all shifts that have a note: {:.1%}".format(notes.shift_id.nunique() / len(shifts)))
print("notes are one per shift:", notes.shift_id.is_unique)

norm = notes.note.map(normalise)
print("blank after normalising:", int((norm == "").sum()), "| 3 characters or fewer:", int((norm.str.len() <= 3).sum()))
print("unique notes (lowercased, punctuation removed):", norm.nunique())
print("columns in the notes file:", list(notes.columns), "-> no hours, date or site; they come from the join")

notes whose shift_id is in shifts.csv: 2117 of 2117
share of all shifts that have a note: 23.9%
notes are one per shift: True
blank after normalising: 69 | 3 characters or fewer: 161
unique notes (lowercased, punctuation removed): 735
columns in the notes file: ['shift_id', 'logged_by', 'note'] -> no hours, date or site; they come from the join


## 2. Collapse the notes to templates
Normalise (lowercase, drop accents and punctuation; no spell-correction, because it mangles Afrikaans and isiZulu), then replace surnames by `NAME` (also with the isiZulu `u` prefix) and times and counts by `#`. Typo variants of the same sentence are then grouped by clustering the character n-grams (2 to 4) of the masked text. This step does not decide any category.

In [3]:
surnames = surname_set(employees)
masked = norm.map(lambda t: mask(t, surnames))
cluster, template = template_clusters(masked)
size = cluster.value_counts()
print("masked unique notes:", masked.nunique(), "| templates (clusters):", template.size)
print("rows in the 10 biggest templates: {:.0%}".format(size.drop(-1, errors="ignore").nlargest(10).sum() / len(notes)))
tmpl = pd.DataFrame({"rows": size.drop(-1, errors="ignore"), "template": template}).dropna().sort_values("rows", ascending=False)
tmpl.head(12)

masked unique notes: 453 | templates (clusters): 68
rows in the 10 biggest templates: 25%


,rows,template
4,62,additional cover requested by site manager signed off
3,60,covering NAME post no show no call
23,57,quiet shift
5,56,took NAME shift as well # posts # guard
28,56,NAME akezanga namhlanje ngimele yena
22,49,handover late agn keys missing
10,49,NAME didnt come in covered the post
50,46,client requested deep clean before the audit approved
53,44,waited # min for handover ob book not signed
43,42,double duty today NAME on family responsibility leave


## 3. The categories
Decided by reading the templates, not assumed beforehand.

| Category | Pile | Meaning |
|---|---|---|
| `client_requested` | client | The client (or its centre or site manager) asked for the extra hours, with or without a recorded approval |
| `client_unconfirmed` | client (unconfirmed) | The client asked, but the note says office approval is not known |
| `relief_no_show` | operational | The next shift or relief did not arrive, or arrived late |
| `colleague_no_show` | operational | A colleague did not come in (no show, absent) and the writer covered |
| `late_handover` | operational | The previous shift handed over late (keys, paperwork, OB book) |
| `equipment_failure` | operational | A machine, lift, generator, gate motor or other equipment failed |
| `colleague_sick_or_leave` | absence, excused or unclear | A colleague was sick, on family leave or at the clinic |
| `stood_in_unspecified` | absence, excused or unclear | The writer covered for someone, no reason given |
| `nothing_useful` | no information | Blank, "ok", "fine", "nothing to report" and the like |

The judgement calls, which are reported as a range in section 8 and not hidden: whether sick and leave cover counts as operational (a staffing problem someone could plan for) or excused, whether "stood in" with no reason is operational or unclear, and whether an unconfirmed client request counts as the client's.

Rule for conflicts: **an operational cue beats a client cue**. "Client signed for the extra hours but real reason is relief no show again" is operational.

## 4. The check: a blind hand-labelled sample
Written **before** either method was built. 205 notes in three samples, drawn with a fixed seed from the raw notes: 120 random (overall accuracy), 40 from minority templates (so the rare categories are tested) and 45 with the heaviest typos (stress test of the typo handling). I labelled them by reading, without any classifier output.

Two honest limits: I, not a client expert, did the labelling, and I had already read a few example notes while looking at the data. The labels and both methods therefore share one person's judgement, so what the check measures is *consistent application of these definitions*, not whether the definitions are right.

In [4]:
blind = pd.read_csv("evaluation/labels/notes_blind_sample.csv", keep_default_na=False)
redrawn = draw_blind_samples(notes, masked, cluster, template)
assert list(redrawn.shift_id) == list(blind.shift_id), "the saved sample is not the seeded draw"
print("blind sample reproduces the seeded draw:", len(blind), "notes")
print(blind.groupby("sample").size().to_dict())
blind.category.value_counts()

blind sample reproduces the seeded draw: 205 notes
{'minority_template': 40, 'random': 120, 'typo_dense': 45}


category
relief_no_show             53
nothing_useful             38
client_requested           23
equipment_failure          22
colleague_sick_or_leave    21
colleague_no_show          18
stood_in_unspecified       16
late_handover              10
client_unconfirmed          4
Name: count, dtype: int64

## 5. Method A: typo-tolerant keyword rules
An ordered list of rules over the normalised note. Each rule is a set of keywords or phrases in English, Afrikaans or isiZulu; a keyword matches if the note contains something close to it (`rapidfuzz`), so `relif`, `hanodver` and `gedk` still hit. Keywords of four letters or fewer must match exactly, because one typo in a short word makes a different word. The order matters: operational cues come first, so a client cue cannot hide a failure. Anything no rule matches goes to a visible `unknown`, never silently into a category.

In [5]:
NOTHING = ["ok", "fine", "all fine", "all good", "all quiet", "quiet shift", "ntr", "n a", "na", "sharp", "nothing to report",
           "no incidents", "no issues on site", "as per normal", "niks om te rapporteer nie", "akukho lutho"]

RULES_V1 = [  # (category, keywords); the first rule that matches wins
    ("relief_no_show",        ["relief", "aflos", "replacement", "akafikanga", "next shift guard"]),
    ("late_handover",         ["handover", "hand over", "oorhandiging"]),
    ("equipment_failure",     ["machine", "masjien", "scrubber", "generator", "lift", "gate motor", "out of order", "kaput", "stukkend", "broke down"]),
    ("colleague_sick_or_leave", ["sick", "siek", "family responsibility", "clinic", "leave", "booked off"]),
    ("colleague_no_show",     ["no show", "didnt come", "did not come", "absent", "akezanga"]),
    ("client_unconfirmed",    ["dont know if office approved"]),
    ("client_requested",      ["client", "klient", "centre manager", "centre management", "site manager", "site mgr", "centre mgr",
                               "requested", "approved", "signed off", "stocktake"]),
    ("stood_in_unspecified",  ["stood in", "covered for", "covering", "took shift", "shift as well"]),
]

def has(text, keyword, tokens):
    if " " in keyword:
        return fuzz.partial_ratio(keyword, text) >= 88
    if len(keyword) <= 4:
        return keyword in tokens
    return bool(tokens) and process.extractOne(keyword, tokens, scorer=fuzz.ratio, score_cutoff=82) is not None

def is_nothing(text):
    if text == "":
        return True
    return len(text.split()) <= 6 and process.extractOne(text, NOTHING, scorer=fuzz.ratio, score_cutoff=85) is not None

def classify_a(text, rules):
    if is_nothing(text):
        return "nothing_useful"
    tokens = text.split()
    for category, keywords in rules:
        if any(has(text, k, tokens) for k in keywords):
            return category
    return "unknown"

pred_a1 = pd.DataFrame({"shift_id": notes.shift_id, "category": [classify_a(t, RULES_V1) for t in norm]})
print(pred_a1.category.value_counts().to_string())

category
nothing_useful             435
client_requested           401
relief_no_show             308
late_handover              212
colleague_no_show          202
equipment_failure          195
colleague_sick_or_leave    190
stood_in_unspecified       133
client_unconfirmed          36
unknown                      5


In [6]:
acc_a1, d_a1 = accuracy_table(blind, pred_a1)
acc_a1.round(3)

,rows,correct,accuracy
sample,,,
minority_template,40.0,40.0,1.000
random,120.0,118.0,0.983
typo_dense,45.0,45.0,1.000
all,205.0,203.0,0.990


In [7]:
# every disagreement with the hand labels, one line per distinct note
bad = d_a1[~d_a1.ok][["note", "category_hand", "category_pred"]]
bad.groupby(["note", "category_hand", "category_pred"]).size().reset_index(name="rows").sort_values("rows", ascending=False)

,note,category_hand,category_pred,rows
0,Nexxt shift guuard did not pitch. Had to cover.,relief_no_show,unknown,1
1,"covering Molefe post, no sow no call..",colleague_no_show,stood_in_unspecified,1


**Pre-fix result (the honest number).** Method A, as first written, scored 99.0% on the blind sample (203 of 205), 98.3% on the random 120. The two misses are heavy typos that the phrase matching was too strict for: `no sow no call` (missed as a no show) and `Nexxt shift guuard` (missed entirely and put in `unknown`). I fix that below by matching each word of a phrase separately and allowing one typo per word (including swapped letters, as in `sikc` or `shfit`). Comparing with Method B (section 7) showed a few more rows with heavy typos (`sikc`, `sic`, `no sow`) that the first fix still missed, and I fixed those too. The final rules are **tuned on this sample and these notes**, so a score of 100% is not an honest estimate; the pre-fix numbers above are.

In [8]:
from rapidfuzz.distance import OSA, Levenshtein

def _word_in(word, tokens):
    # a word matches a token exactly, or with one typo (insert, delete, swap, change) if the word has 5+ letters,
    # or, if it has 4 letters, with one letter dropped, added or swapped with its neighbour (not changed). Words of 3 letters or fewer must match exactly.
    if not tokens:
        return False
    if word in tokens:
        return True
    if len(word) <= 3:
        return False
    for t in tokens:
        if abs(len(t) - len(word)) > 2:
            continue
        if len(word) == 4:
            # one letter dropped, added or two swapped, but not one letter changed (lift/left, sick/sink are different words)
            if OSA.distance(word, t) == 1 and (len(t) != 4 or Levenshtein.distance(word, t) == 2):
                return True
        elif OSA.normalized_similarity(word, t) >= 0.8:
            return True
    return False

def has2(text, keyword, tokens):
    # a phrase matches if every one of its words is found, or the whole phrase is found with a few typos
    return all(_word_in(w, tokens) for w in keyword.split()) or (" " in keyword and fuzz.partial_ratio(keyword, text) >= 90)

def classify_a2(text, rules):
    if is_nothing(text):
        return "nothing_useful"
    tokens = text.split()
    for category, keywords in rules:
        if any(has2(text, k, tokens) for k in keywords):
            return category
    return "unknown"

pred_a2 = pd.DataFrame({"shift_id": notes.shift_id, "category": [classify_a2(t, RULES_V1) for t in norm]})
acc_a2, d_a2 = accuracy_table(blind, pred_a2)
print("unknown after fix:", int((pred_a2.category == "unknown").sum()))
pd.concat({"A v1 (pre-fix, honest)": acc_a1.accuracy, "A v2 (post-fix, tuned)": acc_a2.accuracy}, axis=1).round(3)

unknown after fix: 0


,"A v1 (pre-fix, honest)","A v2 (post-fix, tuned)"
sample,,
minority_template,1.000,1.0
random,0.983,1.0
typo_dense,1.000,1.0
all,0.990,1.0


In [9]:
# does the fix change anything outside the blind sample? (rows where v1 and v2 differ)
chg = pd.DataFrame({"note": notes.note, "v1": pred_a1.category, "v2": pred_a2.category})
chg = chg[chg.v1 != chg.v2]
print("rows changed by the fix:", len(chg), "of", len(notes))
chg.groupby(["v1", "v2"]).size().reset_index(name="rows")

rows changed by the fix: 8 of 2117


,v1,v2,rows
0,stood_in_unspecified,colleague_no_show,3
1,unknown,colleague_sick_or_leave,4
2,unknown,relief_no_show,1


## 6. Method B: unsupervised clusters, labelled once by reading
The templates from section 2 are the clusters. I read each cluster's most common sentence once and gave it a category; every note then takes its cluster's category. This fails differently from the rules: it never misreads a typo (typo variants sit in the same cluster), but a cluster that mixes two meanings takes one label for all of them. The cluster labels were written after the blind sample and without running Method A on the clusters.

In [10]:
CLUSTER_LABELS = {
    "additional cover requested by site manager signed off": "client_requested",
    "covering NAME post no show no call": "colleague_no_show",
    "quiet shift": "nothing_useful",
    "took NAME shift as well # posts # guard": "stood_in_unspecified",
    "NAME akezanga namhlanje ngimele yena": "colleague_no_show",
    "handover late agn keys missing": "late_handover",
    "NAME didnt come in covered the post": "colleague_no_show",
    "client requested deep clean before the audit approved": "client_requested",
    "waited # min for handover ob book not signed": "late_handover",
    "double duty today NAME on family responsibility leave": "colleague_sick_or_leave",
    "NAME off sick again covered": "colleague_sick_or_leave",
    "stood in for NAME": "stood_in_unspecified",
    "fine": "nothing_useful",
    "client asked for extra patrol after the break in tuesday": "client_requested",
    "shift handover delayed by # min": "late_handover",
    "late handover waiting on paperwork": "late_handover",
    "NAME absent took her rounds as well": "colleague_no_show",
    "extra hrs approved by client for the event setup": "client_requested",
    "requested by centre management for load in signed off": "client_requested",
    "worked through NAME at the clinic": "colleague_sick_or_leave",
    "oorhandiging was laat gewag vir sleutels": "late_handover",
    "client asked us to stay for the delivery ok d by centre mgmt": "client_requested",
    "covering for NAME booked off sick": "colleague_sick_or_leave",
    "relief only arrived # stayed until then": "relief_no_show",
    "client wanted extra man on the gate for the event approved": "client_requested",
    "client says stay till # dont know if office approved": "client_unconfirmed",
    "stocktake ran over client asked us to remain they know they pay for it": "client_requested",
    "gedek vir NAME siek gemeld": "colleague_sick_or_leave",
    "generator fault stayed to monitor": "equipment_failure",
    "scrubber broke down had to do the floor manually": "equipment_failure",
    "ntr": "nothing_useful",
    "klient het ekstra ure gevra vir stocktake": "client_requested",
    "control room says relief coming nobody came": "relief_no_show",
    "ok": "nothing_useful",
    "extra patrol per client email approved by office": "client_requested",
    "buffer machine kaput did the floor by hand": "equipment_failure",
    "covered for NAME again 3rd time this month": "stood_in_unspecified",
    "sharp": "nothing_useful",
    "n a": "nothing_useful",
    "next shift akafikanga ngihlale kuze kube #": "relief_no_show",
    "next shift guard did not pitch had to cover": "relief_no_show",
    "machine down agn took twice as long": "equipment_failure",
    "no relief stayed someone must please sort the roster": "relief_no_show",
    "gate motor failed manned it by hand till #": "equipment_failure",
    "double shift because relief never pitched": "relief_no_show",
    "still on site relief was suppose to come #": "relief_no_show",
    "nothing to report": "nothing_useful",
    "all good": "nothing_useful",
    "no replacement sent ngicela sort this out": "relief_no_show",
    "lift out of order everything carried up stairs took long": "equipment_failure",
    "relief no show again": "relief_no_show",
    "no incidents": "nothing_useful",
    "masjien is stukkend alles met die hand gedoen": "equipment_failure",
    "no issues on site": "nothing_useful",
    "aflos het nie opgedaag nie moes aanbly": "relief_no_show",
    "as per normal": "nothing_useful",
    "client signed for the extra hours but real reason is relief no show agn": "relief_no_show",
    "akukho lutho": "nothing_useful",
    "waited for relief nobody came through": "relief_no_show",
    "niks om te rapporteer nie": "nothing_useful",
}
# the one-off clusters are single typo-ridden spellings of sentences above; each takes the label of the sentence it spells
ONE_OFFS = {"balloyi off sick agn covered": "colleague_sick_or_leave", "NAME off sikc again cvered": "colleague_sick_or_leave",
            "stood in for fuorie": "stood_in_unspecified", "stood in for motaug": "stood_in_unspecified", "stood in for zuul": "stood_in_unspecified",
            "shft handover dealyed by # min": "late_handover", "extra hrs apprvoed by clietn for the event setp": "client_requested",
            "tok bothha shift as wlel # posts # guard": "stood_in_unspecified"}
label_by_template = {**CLUSTER_LABELS, **ONE_OFFS}
missing = [t for t in template if t not in label_by_template]
print("clusters without a label:", missing)
cluster_category = cluster.map(lambda c: label_by_template[template[c]] if c >= 0 else "nothing_useful")
pred_b = pd.DataFrame({"shift_id": notes.shift_id, "category": cluster_category.values})
acc_b, d_b = accuracy_table(blind, pred_b)
acc_b.round(3)

clusters without a label: []


,rows,correct,accuracy
sample,,,
minority_template,40.0,40.0,1.0
random,120.0,120.0,1.0
typo_dense,45.0,45.0,1.0
all,205.0,205.0,1.0


In [11]:
bad_b = d_b[~d_b.ok][["note", "category_hand", "category_pred"]]
bad_b.groupby(["note", "category_hand", "category_pred"]).size().reset_index(name="rows").sort_values("rows", ascending=False)

,note,category_hand,category_pred,rows


## 7. Do the methods agree, and how do they compare with the supplied labels?
Where two independent methods disagree is where errors cluster. The supplied file is a third opinion only: it came from someone else's classifier, so agreement with it is not accuracy.

In [12]:
both = pd.DataFrame({"note": notes.note, "A": pred_a2.category, "B": pred_b.category, "supplied": supplied.category})
print("A (pre-fix v1) vs B agree on {:.1%} of {} notes".format((pred_a1.category.values == both.B.values).mean(), len(both)))
print("A (final) vs B agree on {:.1%}  <- A was tuned toward B, so this is no longer independent".format((both.A == both.B).mean()))
print("A vs supplied: {:.1%} | B vs supplied: {:.1%}".format((both.A == both.supplied).mean(), (both.B == both.supplied).mean()))
dis = both[both.A != both.B].groupby(["note", "A", "B"]).size().reset_index(name="rows").sort_values("rows", ascending=False)
print("distinct notes where A and B disagree:", len(dis))
dis.head(25)

A (pre-fix v1) vs B agree on 99.6% of 2117 notes
A (final) vs B agree on 100.0%  <- A was tuned toward B, so this is no longer independent
A vs supplied: 95.9% | B vs supplied: 95.9%
distinct notes where A and B disagree: 0


,note,A,B,rows


In [13]:
# how each method's categories line up with the supplied ones (rows = ours, columns = supplied)
pd.crosstab(both.A, both.supplied)

supplied,client_requested,client_unconfirmed,colleague_no_show,colleague_sick_or_leave,equipment_failure,late_handover,nothing_useful,relief_no_show,stood_in_unspecified
A,,,,,,,,,
client_requested,401,0,0,0,0,0,0,0,0
client_unconfirmed,0,36,0,0,0,0,0,0,0
colleague_no_show,0,0,205,0,0,0,0,0,0
colleague_sick_or_leave,0,0,0,194,0,0,0,0,0
equipment_failure,0,0,0,0,195,0,0,0,0
late_handover,0,0,0,0,0,212,0,0,0
nothing_useful,0,0,0,0,0,0,435,0,0
relief_no_show,0,0,0,0,0,0,0,309,0
stood_in_unspecified,0,0,86,0,0,0,0,0,44


In [14]:
diff_sup = both[both.A != both.supplied].copy()
diff_sup["note_norm"] = diff_sup.note.map(normalise)
diff_sup.groupby(["A", "supplied"]).agg(rows=("note", "size"), example=("note", "first")).sort_values("rows", ascending=False).head(15)

,,rows,example
A,supplied,,
stood_in_unspecified,colleague_no_show,86,"took Jacobs shift as well, 2 posts 1 guard"


**Reading the disagreements.** The two big differences with the supplied labels are not typo problems, they are *definition* choices. A note such as `took Mabaso shift as well, 2 posts 1 guard`, or `covered for Naidoo again, 3rd time this month`, says someone was covered but not why. The supplied labels treat several of these as colleague no-shows (operational), I treat them as `stood_in_unspecified` (reason not stated). Which one is right is a judgement call, and it moves the operational share (section 8 reports both).

In [15]:
# 50 ambiguous notes for you to hand-label (the check I cannot do for you): written once, scored automatically once you fill in `category`
import os
AMBIGUOUS = ["clinic", "took NAME shift", "covered for NAME again", "stood in", "absent took", "family responsibility", "centre manager requested",
             "dont know if office approved", "real reason is relief", "relief only arrived", "siek gemeld", "akezanga"]
path = "evaluation/labels/notes_to_label_by_you.csv"
if not os.path.exists(path):
    rng = np.random.default_rng(1)
    pool = notes[~notes.shift_id.isin(blind.shift_id)].assign(tm=masked)
    picks = []
    for kw in AMBIGUOUS:
        sel = pool[pool.tm.str.contains(kw.replace("took NAME shift", "took NAME shift"), case=False, regex=False)]
        if len(sel) == 0:
            sel = pool[pool.note.str.lower().str.contains(kw.split()[0].lower(), regex=False)]
        picks.append(sel.sample(min(5, len(sel)), random_state=int(rng.integers(1e6))))
    sheet = pd.concat(picks).drop_duplicates("shift_id").head(50)[["shift_id", "note"]].assign(category="")
    sheet.to_csv(path, index=False)
yours = pd.read_csv(path, keep_default_na=False)
filled = yours[yours.category.str.strip() != ""]
print("sheet for you:", len(yours), "notes | labelled so far:", len(filled))
if len(filled):
    mine = pred_a2.set_index("shift_id").category
    print("Method A agrees with your labels on {:.1%}".format((filled.set_index("shift_id").category == mine.reindex(filled.shift_id)).mean()))

sheet for you: 50 notes | labelled so far: 0


**What the check can and cannot show.**
- Both methods score 100% on the blind sample after the fixes, and 99.0% (Method A, before any fix) on the sample as first run. That is an unusually clean result and it has a simple cause: the notes are about 60 sentences repeated with typos, so there is little for a classifier to get wrong.
- The check only shows that the definitions are *applied consistently*. The hand labels, Method A and Method B all encode one person's judgement. The calls that decide the headline are not typo problems: does `took X's shift as well` mean X did not show up, is sick or family leave a failure someone could fix, and is a client's request still the client's when approval is not recorded? The next section reports the split under each reading.
- 100% on this file says nothing about new free text from a supervisor. A new sentence would fall to `unknown` (rules) or be forced into the nearest cluster (Method B).
- The 50-note sheet above is for you to label by hand; the notebook scores Method A against it as soon as `category` is filled in.

## 8. The split: client-asked hours versus operational failures
Notes are the unit here. Section 8.2 weights by hours using the join to `shifts.csv`.

Final category = Method A (the two methods agree on every note); any note A cannot read would take Method B's label, so no note is left unsorted.

In [16]:
final = pred_a2.copy()
unk = final.category == "unknown"
final.loc[unk, "category"] = pred_b.category[unk].values
print("notes where A could not decide and B's label was used:", int(unk.sum()))
assert final.shift_id.is_unique and len(final) == len(notes) and (final.category != "unknown").all()

PILE = {"client_requested": "client asked", "client_unconfirmed": "client asked (approval unknown)",
        "relief_no_show": "operational failure", "colleague_no_show": "operational failure", "late_handover": "operational failure",
        "equipment_failure": "operational failure", "colleague_sick_or_leave": "absence, excused or unclear",
        "stood_in_unspecified": "absence, excused or unclear", "nothing_useful": "no information"}
df = notes.assign(category=final.category.values)
df["pile"] = df.category.map(PILE)
informative = df[df.category != "nothing_useful"]
print("notes:", len(df), "| say nothing useful:", (df.category == "nothing_useful").sum(), "({:.0%})".format((df.category == "nothing_useful").mean()),
      "| informative:", len(informative))
cat_table = df.category.value_counts().rename("notes").to_frame()
cat_table["share of informative"] = (cat_table.notes / len(informative)).where(cat_table.index != "nothing_useful")
cat_table.round(3)

notes where A could not decide and B's label was used: 0
notes: 2117 | say nothing useful: 435 (21%) | informative: 1682


,notes,share of informative
category,,
nothing_useful,435,NaN
client_requested,401,0.238
relief_no_show,309,0.184
late_handover,212,0.126
colleague_no_show,205,0.122
equipment_failure,195,0.116
colleague_sick_or_leave,194,0.115
stood_in_unspecified,130,0.077
client_unconfirmed,36,0.021


In [17]:
# the split under different readings of the judgement calls (share of the informative notes)
c = informative.category.value_counts()
n = len(informative)
OPER = ["relief_no_show", "colleague_no_show", "late_handover", "equipment_failure"]
ABS = ["colleague_sick_or_leave", "stood_in_unspecified"]
rows = []
rows.append(("main (absence cover is its own pile)", c[OPER].sum(), c.get("client_requested", 0), c[ABS].sum(), c.get("client_unconfirmed", 0)))
rows.append(("broad (sick, leave, stood-in = operational)", c[OPER].sum() + c[ABS].sum(), c.get("client_requested", 0), 0, c.get("client_unconfirmed", 0)))
moved = int((df.category.eq("stood_in_unspecified") & supplied.category.eq("colleague_no_show")).sum())
print("stood-in notes (reason not stated) that the supplied labels call colleague no-shows:", moved)
rows.append((f"as the supplied labels read 'took X's shift' / 'covered for X again' ({moved} notes: no-shows)", c[OPER].sum() + moved, c.get("client_requested", 0), c[ABS].sum() - moved, c.get("client_unconfirmed", 0)))
split = pd.DataFrame(rows, columns=["reading", "operational", "client asked", "absence (excused/unclear)", "client asked, approval unknown"]).set_index("reading")
split_pct = split.div(split.sum(axis=1), axis=0) * 100
pd.concat({"notes": split, "% of informative": split_pct.round(1)}, axis=1)

stood-in notes (reason not stated) that the supplied labels call colleague no-shows: 86


notes                                                                       % of informative               \
                                                                                          operational client asked absence (excused/unclear) client asked, approval unknown      operational client asked   
reading                                                                                                                                                                                                     
main (absence cover is its own pile)                                                              921          401                       324                             36             54.8         23.8   
broad (sick, leave, stood-in = operational)                                                      1245          401                         0                             36             74.0         23.8   
as the supplied labels read 'took X's shift' / 'covered for X again' (86 notes: no-shows)        1007          401                       238                             36             59.9         23.8   

                                                                                                                                                    
                                                                                          absence (excused/unclear) client asked, approval unknown  
reading                                                                                                                                             
main (absence cover is its own pile)                                                                           19.3                            2.1  
broad (sick, leave, stood-in = operational)                                                                     0.0                            2.1  
as the supplied labels read 'took X's shift' / 'covered for X again' (86 notes: no-shows)                      14.1                            2.1

In [18]:
# how solid is "the client pays for these"? Approval wording in the client-asked notes
APPROVAL = ["approved", "signed off", "ok d", "okd", "pay for it", "per client email", "approved by office"]
def approval_wording(text):
    t = normalise(text)
    return any(a in t for a in APPROVAL) or any(fuzz.partial_ratio(a, t) >= 88 for a in APPROVAL if len(a) > 6)
cr = df[df.category == "client_requested"].copy()
cr["has_approval_wording"] = cr.note.map(approval_wording)
print("client_requested notes:", len(cr), "| with approval or sign-off wording:", int(cr.has_approval_wording.sum()),
      "| with none:", int((~cr.has_approval_wording).sum()), "({:.0%})".format((~cr.has_approval_wording).mean()))
print("so for these the client paying is the supervisor's claim, not a record of approval")
print(cr[~cr.has_approval_wording].note.map(normalise).map(lambda t: mask(t, surnames)).value_counts().head(5).to_string())

# disguised failures: a client cue and a failure cue in the same note
disguised = df[df.note.map(normalise).str.contains("client", na=False) & df.category.isin(OPER)]
print("\nnotes that mention the client but are sorted as an operational failure (the real reason won):", len(disguised))

client_requested notes: 401 | with approval or sign-off wording: 290 | with none: 111 (28%)
so for these the client paying is the supervisor's claim, not a record of approval
note
client asked for extra patrol after the break in tuesday    25
klient het ekstra ure gevra vir stocktake                   22
centre manager requested additional cover for stocktake     16
centre mgr requested additional cover for stocktake          7
kliennt het ekstra ure gevra vir stocktake                   2

notes that mention the client but are sorted as an operational failure (the real reason won): 19


### 8.2 Weighted by hours
A note is about one shift, and the overtime is a weekly figure, so there is no exact way to say how many *overtime hours* a note explains. Four weights are shown, from simplest to closest to overtime: the number of notes; the hours of the noted shift; the hours of that shift above the usual 9-hour shift (45 hours over 5 days); and the hours of noted shifts that fall in a week where that employee ID worked more than 45 hours. Hours use the greater-portion attribution from the main analysis.

In [19]:
cutoff = shifts.shift_date.max()
current_week = cutoff - pd.Timedelta(days=cutoff.weekday())
shift_hours = segments.groupby("shift_id").agg(hours=("hours", "sum"), week_start=("week_start", "first"), attributed_date=("attributed_date", "first"))
week_total = segments.groupby(["employee_id", "week_start"]).hours.sum().rename("week_hours")
info = (df.merge(shifts[["shift_id", "employee_id", "site_id", "shift_date"]], on="shift_id", how="left")
          .merge(shift_hours, on="shift_id", how="left")
          .merge(week_total, on=["employee_id", "week_start"], how="left"))
info["hours"] = info.hours.fillna(0)
info["extra_over_9"] = (info.hours - 9).clip(lower=0)
info["in_week_over_45"] = (info.week_hours > 45) & (info.week_start < current_week)
info["hours_in_week_over_45"] = info.hours.where(info.in_week_over_45, 0)
print("notes with a shift that has counted hours:", int((info.hours > 0).sum()), "of", len(info), "(the rest have no clock-out)")

inf = info[info.category != "nothing_useful"]
def pile_shares(weight):
    g = inf.groupby("pile")[weight].sum() if weight != "n" else inf.groupby("pile").size()
    return (g / g.sum() * 100).round(1)
shares = pd.DataFrame({"notes": pile_shares("n"), "hours of the shift": pile_shares("hours"),
                       "hours above 9": pile_shares("extra_over_9"), "hours in weeks over 45": pile_shares("hours_in_week_over_45")})
display(shares)
print("total hours on noted shifts: {:.0f} | above 9: {:.0f} | in weeks over 45: {:.0f}".format(inf.hours.sum(), inf.extra_over_9.sum(), inf.hours_in_week_over_45.sum()))

notes with a shift that has counted hours: 2044 of 2117 (the rest have no clock-out)


,notes,hours of the shift,hours above 9,hours in weeks over 45
pile,,,,
"absence, excused or unclear",19.3,19.2,19.0,19.6
client asked,23.8,24.0,23.8,25.1
client asked (approval unknown),2.1,2.0,1.6,1.8
operational failure,54.8,54.7,55.6,53.5


total hours on noted shifts: 18492 | above 9: 4064 | in weeks over 45: 6532


In [20]:
# per category, in hours above the usual 9-hour shift
(inf.groupby("category").agg(notes=("shift_id", "size"), shift_hours=("hours", "sum"), hours_above_9=("extra_over_9", "sum"), avg_shift=("hours", "mean"))
    .sort_values("hours_above_9", ascending=False).round(1))

,notes,shift_hours,hours_above_9,avg_shift
category,,,,
client_requested,401,4438.8,968.2,11.1
relief_no_show,309,3545.0,917.0,11.5
equipment_failure,195,2148.2,477.2,11.0
colleague_no_show,205,2186.8,469.0,10.7
colleague_sick_or_leave,194,2136.5,457.8,11.0
late_handover,212,2240.0,395.5,10.6
stood_in_unspecified,130,1423.0,313.5,10.9
client_unconfirmed,36,374.0,65.5,10.4


## 9. Where is it concentrated?
Two pitfalls: with 24 supervisors (or 30 surnames, or a handful of sites) the biggest-looking one is expected to look odd by chance, so every comparison is tested against what chance would give and corrected for the number of comparisons (Bonferroni). And "no meaningful concentration" is a valid finding: it points to a systemic cause, not to a few people.

In [21]:
inf = inf.copy()
inf["operational"] = inf.category.isin(OPER)
overall = inf.operational.mean()
sup = inf.groupby("logged_by").agg(notes=("operational", "size"), operational=("operational", "sum"))
sup["share"] = sup.operational / sup.notes
sup["p"] = [stats.binomtest(int(k), int(m), overall).pvalue for k, m in zip(sup.operational, sup.notes)]
sup["p_bonferroni"] = (sup.p * len(sup)).clip(upper=1)
ct = pd.crosstab(inf.logged_by, inf.pile)
chi2, p_chi, dof, _ = stats.chi2_contingency(ct)
print("overall operational share: {:.1%}".format(overall))
print("do supervisors differ in their mix of reasons at all? chi-square p = {:.2f} ({} supervisors)".format(p_chi, len(sup)))
print("supervisors with a significant operational share after Bonferroni:", int((sup.p_bonferroni < 0.05).sum()))
sup.sort_values("share", ascending=False).head(5).round(3)

overall operational share: 54.8%
do supervisors differ in their mix of reasons at all? chi-square p = 0.77 (24 supervisors)
supervisors with a significant operational share after Bonferroni: 0


,notes,operational,share,p,p_bonferroni
logged_by,,,,,
SUP-03,63,43,0.683,0.032,0.763
SUP-23,75,51,0.680,0.027,0.644
SUP-14,70,45,0.643,0.119,1.000
SUP-13,63,39,0.619,0.311,1.000
SUP-17,79,48,0.608,0.310,1.000


In [22]:
# guards: which colleague is named in the absence notes (surname only: several employees share a surname)
surname_list = sorted(surnames)
def named_surname(text):
    for tok in normalise(text).split():
        t = tok[1:] if tok.startswith("u") and tok[1:] in surnames else tok
        if t in surnames:
            return t
        if len(tok) >= 5:
            hit = process.extractOne(t, surname_list, scorer=OSA.normalized_similarity, score_cutoff=0.78)
            if hit:
                return hit[0]
    return None
ab = inf[inf.category.isin(["colleague_no_show", "colleague_sick_or_leave", "stood_in_unspecified"])].copy()
ab["surname"] = ab.note.map(named_surname)
print("absence notes:", len(ab), "| with a surname found:", int(ab.surname.notna().sum()))
last = employees.full_name.map(lambda n: normalise(n.split()[-1])).value_counts()
obs = ab.surname.value_counts().reindex(last.index).fillna(0)
exp = last / last.sum() * obs.sum()
chi_g, p_g = stats.chisquare(obs, exp)
guard = pd.DataFrame({"observed": obs, "expected": exp.round(1)})
guard["p"] = [stats.binomtest(int(o), int(obs.sum()), e / obs.sum()).pvalue for o, e in zip(guard.observed, exp)]
guard["p_bonferroni"] = (guard.p * len(guard)).clip(upper=1)
print("do some surnames appear more than their share of employees would give? chi-square p = {:.3f}".format(p_g))
print("surnames significant after Bonferroni ({} comparisons):".format(len(guard)), list(guard[guard.p_bonferroni < 0.05].index))
guard.sort_values("observed", ascending=False).head(6)

absence notes: 529 | with a surname found: 523
do some surnames appear more than their share of employees would give? chi-square p = 0.212
surnames significant after Bonferroni (30 comparisons): []


,observed,expected,p,p_bonferroni
full_name,,,,
sibiya,36,27.0,0.091687,1.0
fourie,28,24.6,0.468278,1.0
wyk,28,24.6,0.468278,1.0
molefe,27,24.6,0.604383,1.0
botha,23,17.2,0.174769,1.0
jacobs,23,22.1,0.827468,1.0


In [23]:
# sites and weekdays (possible because each note joins to its shift)
sites = pd.read_csv("data/sites.csv").set_index("site_id")["site_name"]
site = inf.groupby("site_id").agg(notes=("operational", "size"), operational=("operational", "sum"))
site["share"] = site.operational / site.notes
site["p_bonferroni"] = [min(1, stats.binomtest(int(k), int(m), overall).pvalue * len(site)) for k, m in zip(site.operational, site.notes)]
site.insert(0, "site", site.index.map(sites))
shifts_per_site = shifts.groupby("site_id").size()
site["notes_per_100_shifts"] = (df.merge(shifts[["shift_id", "site_id"]], on="shift_id").groupby("site_id").size() / shifts_per_site * 100).round(1)
display(site.round(3))
sid = df.merge(shifts[["shift_id", "site_id"]], on="shift_id").site_id
noted_tab = pd.DataFrame({"noted": sid.value_counts(), "all": shifts.groupby("site_id").size()})
noted_tab["not_noted"] = noted_tab["all"] - noted_tab.noted
print("do sites differ in how often a shift gets a note? chi-square p = {:.3f}".format(stats.chi2_contingency(noted_tab[["noted", "not_noted"]])[1]))
wd = inf.assign(weekday=inf.shift_date.dt.day_name()).groupby("weekday").agg(notes=("operational", "size"), operational_share=("operational", "mean")).reindex(
    ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"])
chi_w = stats.chi2_contingency(pd.crosstab(inf.shift_date.dt.day_name(), inf.operational))[1]
print("weekday vs operational share: chi-square p = {:.2f}".format(chi_w))
wd.round(3)

,site,notes,operational,share,p_bonferroni,notes_per_100_shifts
site_id,,,,,,
ST-01,Sandton City Retail,206,107,0.519,1.000,19.2
ST-02,OR Tambo Cargo Terminal,243,143,0.588,1.000,24.1
ST-03,Menlyn Park Centre,344,189,0.549,1.000,26.0
ST-04,Century City Offices,289,146,0.505,0.937,22.3
ST-05,Durban Point Depot,338,190,0.562,1.000,25.9
ST-06,Gqeberha Plant,262,146,0.557,1.000,26.3


do sites differ in how often a shift gets a note? chi-square p = 0.000
weekday vs operational share: chi-square p = 0.61


,notes,operational_share
weekday,,
Monday,246,0.504
Tuesday,254,0.579
Wednesday,238,0.559
Thursday,212,0.557
Friday,237,0.515
Saturday,235,0.553
Sunday,260,0.565


## 10. Do noted shifts and breach weeks look different?
Notes exist for about a quarter of shifts, so the split above describes noted shifts only. Two quick checks of how much that matters, and a link back to the prediction task.

In [24]:
all_shift = shifts[~shifts.excluded].merge(shift_hours.reset_index()[["shift_id", "hours", "week_start"]], on="shift_id")
all_shift = all_shift.merge(week_total, on=["employee_id", "week_start"], how="left")
all_shift["noted"] = all_shift.shift_id.isin(notes.shift_id)
cmp_ = all_shift.groupby("noted").agg(shifts=("shift_id", "size"), avg_hours=("hours_y", "mean") if "hours_y" in all_shift else ("hours", "mean"))
cmp_["share in a week over 45h"] = all_shift[all_shift.week_start < current_week].groupby("noted").apply(lambda g: (g.week_hours > 45).mean())
cmp_["share in a week over 55h (breach)"] = all_shift[all_shift.week_start < current_week].groupby("noted").apply(lambda g: (g.week_hours > 55).mean())
display(cmp_.round(3))

inf2 = inf[inf.week_start < current_week].copy()
inf2["breach_week"] = inf2.week_hours > 55
tab = pd.crosstab(inf2.breach_week, inf2.pile, normalize="index").round(3) * 100
print("pile mix on noted shifts that fall in a breach week (True) vs other weeks (False), % of notes:")
display(tab)
print("chi-square p = {:.2f} (breach-week notes: {})".format(stats.chi2_contingency(pd.crosstab(inf2.breach_week, inf2.pile))[1], int(inf2.breach_week.sum())))

,shifts,avg_hours,share in a week over 45h,share in a week over 55h (breach)
noted,,,,
False,6635,9.354,0.371,0.059
True,2044,11.143,0.370,0.067


pile mix on noted shifts that fall in a breach week (True) vs other weeks (False), % of notes:


pile,"absence, excused or unclear",client asked,client asked (approval unknown),operational failure
breach_week,,,,
False,19.5,23.6,2.2,54.7
True,15.0,30.8,1.9,52.3


chi-square p = 0.33 (breach-week notes: 107)


## 11. Output: `note_classifications.csv`
One row per note, exactly the columns `shift_id,category,note`.

In [25]:
out = pd.DataFrame({"shift_id": notes.shift_id, "category": final.category.values, "note": notes.note})
assert list(out.columns) == ["shift_id", "category", "note"] and len(out) == 2117 and out.shift_id.is_unique and out.category.notna().all()
assert out.note.equals(notes.note)
out.to_csv("note_classifications.csv", index=False)
print("written:", out.shape, "| categories:", out.category.nunique())
out.category.value_counts().to_frame("notes")

written: (2117, 3) | categories: 9


,notes
category,
nothing_useful,435
client_requested,401
relief_no_show,309
late_handover,212
colleague_no_show,205
equipment_failure,195
colleague_sick_or_leave,194
stood_in_unspecified,130
client_unconfirmed,36


## Summary
**The sorting.** 2,117 notes collapse to about 60 repeated sentences (typos, surnames and times aside). 435 notes (21%) say nothing useful; the other 1,682 are sorted into 8 reasons: client asked 24% (plus 2% where the client asked but office approval is unknown), relief no-show or late 18%, late handover 13%, colleague no-show 12%, equipment failure 12%, colleague sick or on leave 12%, stood in with no reason given 8%. No language model was used (few templates, no independent labels to train on).

**The split.** Of the notes that say something, **55% are operational failures** (no-shows, late handovers, broken equipment), **24% are hours the client asked for**, **19% are cover for an absence** where the cause is excused or not stated, and 2% are client requests with unknown approval. The operational share depends on judgement calls and is reported as a range:
- 55% if sick/leave and "stood in" cover is its own pile (the main reading);
- 60% if the 86 "took X's shift as well" and "covered for X again" notes count as no-shows (how the supplied labels read them);
- 74% if all absence cover counts as fixable.
The client share stays at 24 to 26% under every reading. Inside the operational pile, staffing dominates: relief no-shows and colleague no-shows are 56% of it, late handovers 23%, equipment 21%. Weighting by hours (shift hours, hours above 9, hours in weeks over 45) moves every pile by less than two points, so the notes-based split is not an artefact of counting notes.

**Two things that weaken "the client pays for those hours".** 19 notes say the client signed or asked but give a failure as the real reason (counted as operational), and 111 of the 401 client-asked notes (28%) contain no approval or sign-off wording at all, so there the client paying is the supervisor's claim.

**Where it is concentrated: it is not.** The mix of reasons does not differ between the 24 supervisors (chi-square p = 0.77). SUP-03 looks worst (68% operational) but that is p = 0.76 after correcting for 24 comparisons. No guard surname is named more often than its share of employees would give (p = 0.21 overall, none significant after correction), no site differs (all corrected p above 0.9), and neither does the weekday (p = 0.61) or a breach week against other weeks (p = 0.33, only 107 notes). This points to a systemic cause (relief and staffing) rather than a few supervisors or sites. With about 90 notes per supervisor only large differences could be detected, so "no evidence" is not "proven equal". Two other facts about the notes: they cover only 24% of shifts and those shifts are longer (11.1 hours against 9.4), so they describe the long shifts; and sites differ in how often they write a note (19 to 26 per 100 shifts), which is a documentation difference, not a difference in causes.

**Does the sorting work?** Against a blind hand-labelled sample of 205 notes (written before either method): Method A as first written scored 99.0% (98.3% on the random 120), Method B (clusters, labelled once by reading) 100%. The two misses were heavy typos. I then fixed the rules, which makes their 100% a tuned number, not an estimate. Before the fix the two independent methods agreed on 99.6% of all notes. The supplied labels agree with mine on 95.9%, and every disagreement is the same definition choice (86 stood-in notes), not a typo.

**What the check cannot tell you.** The hand labels, Method A and Method B all encode one person's judgement, so the check shows consistent application of the definitions, not that the definitions are right. The calls that move the headline are sick versus failure, "took X's shift" as a no-show or not, and unconfirmed client requests. 100% on this file says nothing about new free text: the notes are templated, and a new sentence would end up as `unknown`. `evaluation/labels/notes_to_label_by_you.csv` has 50 of the ambiguous notes for you to label by hand; filling in `category` makes the notebook score Method A against your judgement, which is the most useful extra check.

**Outputs.** `note_classifications.csv` (2,117 rows, `shift_id,category,note`).